# Can the mtanh baseline fit be trusted where the scan acts?

The sparse-grid axes are scale factors on mtanh fit parameters, so a bad fit
does not produce a bad point — it silently redefines what every axis value
means, for the whole scan. This notebook answers one question: **is each fit
good enough, where the scan acts, to license those scale factors?**

Scaling moved out. Bounds conversion, the scan box and the scaled-profile
gallery are now `profile_scaling/profile_scaling.ipynb`. Both notebooks import
`pedestal_scan.py` at the repo root, which owns the discharge list, the fit
settings, the pedestal-region rule, the axes and the literature bounds — one
source of truth, so the two cannot drift into disagreeing.

**One parameterization: `fit_mtanh_full`** (Stefanikova 2016, axis-to-SOL in one
formula), driven by `apply_mtanh_full`. The pedestal-only Bruncrona form lost on
every discharge that separated them — 129038 `ne` 13.3% vs 0.6% — and one knob
set across every axis and discharge is worth more to a campaign than a per-axis
best form.

**The verdict window is per discharge, and measured.** A single global window is
the wrong test twice over: `rms_relative` over the whole profile scores a fit on
core structure the scan never touches, and a fixed 0.60–0.95 band asks 132588
(whose $p_e$ is already falling at $\rho_t\approx0.56$) and 129015 (0.86) the
same question. Each discharge gets the quarter-maximum band of $|\nabla p_e|$
**taken from the data** — not from the fit, which is the thing under test —
widened to cover the radii GENE is run at.

**Two numbers, not one.** `rms_win` licenses the scale factors. But
`apply_mtanh_full` replaces the *whole* profile and CHEASE-BS reshapes on all of
it, so the full-radius picture matters too — that is the mistake that got an
earlier tuning pass reverted after it won in-window and wandered in the core.

In [ ]:
import sys, json, pathlib
import numpy as np
import matplotlib.pyplot as plt

# repo root, wherever this is checked out
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "pedestal_scan.py").exists())
sys.path.insert(0, str(ROOT))

from pedestal_scan import (Campaign, PLOT_VARS, COL, TRUST, GRAD_SEARCH,
                           ANALYSIS_RADII, FIT_KWARGS, window_stats)

camp = Campaign()                      # loads and fits all four discharges
shots = camp.shots
varlist = PLOT_VARS

print(f"fit settings: {FIT_KWARGS}   trust threshold: {TRUST:.0%} of range")
print(f"{'shot':>7}  {'pedestal region':>17} {'peak':>6}  {'verdict window':>15}"
      f"  {'GENE radii':>13}")
print("-" * 70)
for s in shots:
    d = camp[s]
    lo, pk, hi = d.ped
    print(f"{s:>7}  {lo:>8.3f}-{hi:<8.3f} {pk:>6.3f}  "
          f"{d.win[0]:>7.3f}-{d.win[1]:<7.3f}  "
          + ",".join(f"{r:.3f}" for r in ANALYSIS_RADII[s]).rjust(13))
print("\npedestal region = quarter-max band of |grad pe| from the data; verdict "
      "window = that band widened to the GENE radii and to 0.95, capped at 0.99")

## The verdict — error inside each discharge's own window

`rms_win` / `max_win` are taken over the verdict window; `rms_ped` over the
pedestal region alone; `rms_in` over the stretch immediately inboard of it (0.50
to the window's inner edge); `rms_global` is the whole-profile number, kept for
contrast. Where `rms_global` and `rms_win` diverge, the global number was
measuring something the scan does not touch.

`rms_in` is not part of the verdict but is the honest check on it: a profile
passing in-window while carrying several percent just inboard — 129015 `Ti` — is
being flattered by a narrow window, and for `Ti`/`ni` that error still reaches
CHEASE through the closure.

`b_pos` on 0.85 is the fitter's lower bound, not a located pedestal. Such a fit
can score well and still be the wrong shape to scale; it is the cause of
132588's 35.7 %$\psi_N$ `ne` width next door.

In [ ]:
rows = []
for shot in shots:
    d = camp[shot]
    x, win, ped = d.x, d.win, (d.ped[0], d.ped[2])
    inb = (GRAD_SEARCH[0], win[0])
    for var in varlist:
        f = d.var[var]
        if "err" not in f:
            rows.append({"shot": shot, "var": var, "note": f.get("error", "")[:40],
                         **{k: np.nan for k in ("rms_win", "max_win", "rms_ped",
                                                "rms_in", "rms_global", "b_pos")},
                         "pinned": False, "ok": False})
            continue
        rms, mx = window_stats(x, f["err"], win)
        rows.append({
            "shot": shot, "var": var, "rms_win": rms, "max_win": mx,
            "rms_ped": window_stats(x, f["err"], ped)[0],
            "rms_in": (window_stats(x, f["err"], inb)[0] if win[0] > inb[0]
                       else np.nan),
            "rms_global": f["rms_global"], "b_pos": f["params"]["b_pos"],
            "pinned": f["pinned"], "ok": rms <= TRUST, "note": ""})

hdr = (f"{'shot':>7} {'var':<4} {'rms_win':>8} {'max_win':>8} {'rms_ped':>8} "
       f"{'rms_in':>7} {'rms_global':>10} {'b_pos':>7} {'verdict':>8}")
print(hdr); print("-" * len(hdr))
pct = lambda v, w: (f"{v*100:>{w-1}.2f}%" if v == v else f"{'--':>{w}}")
for r in rows:
    print(f"{r['shot']:>7} {r['var']:<4} {pct(r['rms_win'],8)} {pct(r['max_win'],8)} "
          f"{pct(r['rms_ped'],8)} {pct(r['rms_in'],7)} {pct(r['rms_global'],10)} "
          f"{r['b_pos']:>7.4f} {'TRUST' if r['ok'] else 'reject':>8}"
          + ("   b_pos ON BOUND" if r["pinned"] else "")
          + (f"   {r['note']}" if r["note"] else ""))

print(f"\nTRUST: rms inside the discharge's verdict window <= {TRUST:.0%} of range")
print(f"{'shot':>7}  {'Te':>7} {'ne':>7} {'pe':>7}   axes usable?")
print("-" * 48)
verdict = {}
for shot in shots:
    v = {var: next(r["rms_win"] for r in rows
                   if r["shot"] == shot and r["var"] == var)
         for var in ("Te", "ne", "pe")}
    ok = all(t == t and t <= TRUST for t in v.values())
    verdict[shot] = {**{f"{k}_rms_win": t for k, t in v.items()},
                     "window": list(camp[shot].win),
                     "ped_region": [camp[shot].ped[0], camp[shot].ped[2]],
                     "ok": ok}
    print(f"{shot:>7}  " + " ".join(f"{t*100:>6.2f}%" for t in v.values())
          + f"   {'yes' if ok else 'NO — refit before scaling'}")

with open("mtanh_fit_quality.json", "w") as fh:
    json.dump({"form": "full", "trust_threshold": TRUST, "rows": rows,
               "verdict": verdict}, fh, indent=1, default=str)
print("\nwritten: mtanh_fit_quality.json")

## Look at it

Three figures, in reading order.

1. **Pedestal region** — data vs fit over each discharge's own window (shaded =
   verdict window, dotted = GENE radii, dash-dot = peak $|\nabla p_e|$). What
   disqualifies a fit is error *rising inside the shaded band*.
2. **Full radius** — the same fits axis to SOL. This is the profile
   `apply_mtanh_full` writes and CHEASE-BS reshapes on all of it, so a fit that
   nails the pedestal and wanders on axis is a defect the window score cannot
   fail. Watch for a core missing the measured on-axis value, a non-monotonic
   bump, or the fit crossing the data inboard of the pedestal top. Each panel
   prints `win` and `all` so the two views cannot be read apart.
3. **The verdict** — one bar per profile, threshold drawn.

Bottom row of each grid is the residual against radius, always full radius.
Core error alone is not disqualifying for the *scale factors* — no axis acts
there — but it is what CHEASE-BS sees.

In [ ]:
def profile_grid(xlim, title):
    """Data vs fit; xlim=None zooms on each discharge's window, (0,1) draws all."""
    fig, axes = plt.subplots(len(PLOT_VARS) + 1, len(shots),
                             figsize=(3.5 * len(shots), 2.2 * (len(PLOT_VARS) + 1)),
                             squeeze=False)
    for j, shot in enumerate(shots):
        d = camp[shot]
        x, win = d.x, d.win
        lo, pk, hi = d.ped
        xl = xlim or (win[0] - 0.15, 1.0)
        for i, var in enumerate(PLOT_VARS):
            ax = axes[i][j]
            f = d.var[var]
            m = (x >= xl[0]) & (x <= xl[1])
            ax.plot(x[m], f["y"][m], ".", ms=3, color="0.55", label="data")
            if "yhat" in f:
                ax.plot(x[m], f["yhat"][m], "-", lw=1.5, color=COL[var],
                        label="full fit")
                r = window_stats(x, f["err"], win)[0]
                ax.text(0.03, 0.08,
                        f"win {r*100:.2f}%  |  all {f['rms_global']*100:.2f}%",
                        fontsize=7, transform=ax.transAxes,
                        color="green" if r <= TRUST else "tab:red")
            ax.axvspan(*win, color="tab:green", alpha=0.10)
            ax.axvline(pk, color="k", lw=0.7, ls="-.", alpha=0.5)
            for r0 in ANALYSIS_RADII[shot]:
                ax.axvline(r0, color="k", lw=0.8, ls=":", alpha=0.8)
            ax.set_xlim(*xl); ax.tick_params(labelsize=7)
            if i == 0:
                ax.set_title(f"{shot}   ped {lo:.2f}-{hi:.2f}", fontsize=10)
            if j == 0:
                ax.set_ylabel(var)
        ax = axes[-1][j]
        for var in varlist:
            f = d.var[var]
            if "err" in f:
                ax.plot(x, 100 * f["err"], lw=1.1, color=COL[var], label=var,
                        alpha=0.9)
        ax.axvspan(*win, color="tab:green", alpha=0.10)
        ax.axhline(0, color="k", lw=0.6)
        ax.set_xlim(0, 1.0); ax.set_ylim(-8, 8)
        ax.set_xlabel("rho_tor"); ax.tick_params(labelsize=7)
        if j == 0:
            ax.set_ylabel("(fit - data)/range [%]")
    axes[0][-1].legend(fontsize=6)
    axes[-1][-1].legend(fontsize=6, ncol=2)
    fig.suptitle(title)
    plt.tight_layout()
    return fig


profile_grid(None, "Pedestal region — shaded: verdict window, dotted: GENE "
                   "radii, dash-dot: peak |grad pe|")
plt.show()
profile_grid((0.0, 1.0), "Full radius — what apply_mtanh_full hands to CHEASE-BS")
plt.show()

fig, ax = plt.subplots(figsize=(10, 3.4))
ax.bar(np.arange(len(rows)), [100 * r["rms_win"] for r in rows], 0.6,
       color=[COL[r["var"]] for r in rows])
ax.axhline(100 * TRUST, color="k", ls="--", lw=1.2, label=f"trust {TRUST:.0%}")
ax.set_xticks(np.arange(len(rows)))
ax.set_xticklabels([f"{r['shot']} {r['var']}" for r in rows], fontsize=6,
                   rotation=90)
ax.set_yscale("log"); ax.set_ylabel("rms in verdict window [% of range]")
ax.legend(fontsize=8)
ax.set_title("per-discharge verdict — below the dashed line is usable")
plt.tight_layout(); plt.show()

### If a profile is rejected

In order of effort; the verdict table is the arbiter.

1. **Raise `pedestal_weight`** (currently 8) — buys pedestal accuracy at the
   core's expense, which is the trade this window says we want.
2. **Lower `ped_threshold`** (default 0.85) — for the discharges whose `b_pos`
   is pinned on that bound the pedestal genuinely sits further in and the fitter
   is not allowed to look there. First thing to try for 129038 and 132588.
3. **Supply explicit `p0`** — seed `b_pos` and `b_width` from the measured
   peak-gradient location printed above; those two are what the auto-guess gets
   wrong.

Change these in `pedestal_scan.FIT_KWARGS` so the scaling notebook sees the same
fits, never inline in one notebook.

**Tried and rejected (2026-08-22): automated per-profile tuning** of
`ped_threshold`, seeded `p0` and freed `b_pos` bounds, swept over
`pedestal_weight` 8–64. It worked on the metric this notebook reports — every
profile passed, worst rms 0.74%, 129038 `pe` 4.54% → 0.21% — and was reverted
anyway. Freeing `b_pos` inward (0.68–0.76) buys that by letting the mtanh
reinterpret half the profile as pedestal, and the reconstruction is distorted
outside the pedestal. **CHEASE-BS consumes the whole profile, not this window**,
so a fit that wins in-window and wanders in the core is worse for the campaign
than one that is mediocre in-window and stable everywhere.